In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers
from PIL import Image, ImageDraw
from tqdm import tqdm
import random
import os
import xml.etree.ElementTree as ET

In [ ]:
path_images = "cat_dog_detection/images"
path_annots = "cat_dog_detection/annotations"

In [ ]:
def read_data(path_images, path_annots):
    img_paths = []
    labels = []
    coords = []
    for ann in tqdm(os.listdir(path_annots)):
        ann_path = os.path.join(path_annots, ann)
        tree = ET.parse(ann_path)
        img_paths.append(os.path.join(path_images, tree.find("filename").text))
        labels.append(tree.find("object/name").text)
        x_min, x_max, y_min, y_max = (int(tree.find("object/bndbox/xmin").text),
                                      int(tree.find("object/bndbox/xmax").text), 
                                      int(tree.find("object/bndbox/ymin").text), 
                                      int(tree.find("object/bndbox/ymax").text))
        coords.append([x_min, y_min, x_max, y_max])
    return img_paths, labels, coords

In [ ]:
img_paths, labels, coords = read_data(path_images, path_annots)

In [ ]:
x_train, x_test, lab_train, lab_test, coords_train, coords_test = train_test_split(img_paths, labels, coords, test_size=0.15, random_state=42)

In [ ]:
classes = {"cat": 0, "dog": 1}

In [ ]:
plt.figure(figsize=(15, 25))
index = random.sample(range(len(img_paths)), k=50)
ind = 0
for i in range(50):
    plt.subplot(10, 5, i + 1)
    img = Image.open(img_paths[ind]).convert("RGB")
    w, h = img.size
    img = img.resize((224, 224))
    x_min, y_min, x_max, y_max = coords[ind]
    new_x_min, new_y_min, new_x_max, new_y_max = round(x_min / w * 224), round(y_min / h * 224), round(x_max / w * 224), round(y_max / h * 224)
    draw = ImageDraw.Draw(img)
    draw.rectangle([new_x_min, new_y_min, new_x_max, new_y_max], outline="blue", width=5)
    lab = labels[ind]
    img = np.asarray(img)
    plt.imshow(img)
    plt.title(lab)
    plt.xticks([])
    plt.yticks([])
    ind += 1

In [ ]:
class MyDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_paths, labels, coords, classes, batch_size=32, target_size=(224, 224), one_hot=True):
        self.image_paths = image_paths
        self.labels = labels
        self.coords = coords
        self.classes = classes
        self.batch_size = batch_size
        self.target_size = target_size
        self.one_hot = one_hot
    
    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_paths = self.image_paths[index * self.batch_size : (index + 1) * self.batch_size]
        batch_labels = self.labels[index * self.batch_size : (index + 1) * self.batch_size]
        batch_coords = self.coords[index * self.batch_size : (index + 1) * self.batch_size]

        X = np.zeros((len(batch_paths), *self.target_size, 3), dtype="float32")
        y_labels = np.array([self.classes[i] for i in batch_labels])
        y_coords = []

        for i, path in enumerate(batch_paths):
            img = Image.open(path).convert("RGB")
            w, h = img.size
            img = img.resize(self.target_size)
            X[i] = tf.keras.applications.densenet.preprocess_input(np.asarray(img))
            x_min, y_min, x_max, y_max = batch_coords[i]
            new_x_min, new_y_min, new_x_max, new_y_max = round(x_min / w * 224), round(y_min / h * 224), round(x_max / w * 224), round(y_max / h * 224)
            y_coords.append([new_x_min, new_y_min, new_x_max, new_y_max])

        y_coords = np.array(y_coords)
        y_coords = tf.keras.applications.densenet.preprocess_input(y_coords)

        if self.one_hot:
            y_labels = tf.keras.utils.to_categorical(y_labels)

        return X, y_labels, y_coords

In [ ]:
def reverse_densenet_preprocess(x):
    img = np.array(x, dtype=np.float32).copy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])    
    img = (img * std + mean) * 255.0
    return np.round(img).astype(int)

In [ ]:
train = MyDataGenerator(x_train, lab_train, coords_train, classes)
test = MyDataGenerator(x_test, lab_test, coords_test, classes)